<a href="https://colab.research.google.com/github/comsadavid/AI-solutions/blob/main/competitions/mlcompete/Robo%C8%9Bi_%C3%AEn_labirint_9_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

In [ ]:
df_train

,robot_id,arena_type,weather,difficulty,laps,avg_speed_mps,distance_m,battery_used_pct,collisions,unique_cells,items_collected,pause_count,turn_rate,scan_accuracy,return_time_s,efficiency_score,coverage_ratio,scan_turn_sync,pickup_efficiency,detour_index,safety_margin,patrol_consistency,speed_burst,risk_index,strategy_label
0,TR_0000,urban,clear,2,3,1.561222,2999.630719,72.252973,7,247.530750,5,1,0.681408,0.917439,482.744368,13.290300,6.719937,33.092663,4.839689,-10.000000,10.983818,-0.176148,9.821879,7.942570,explorer
1,TR_0001,urban,clear,1,5,1.442525,3154.068573,64.676535,6,334.130274,0,2,0.776598,0.799059,432.603240,26.398660,5.192224,30.436153,1.095235,-10.000000,9.600871,3.270004,9.372641,3.101063,explorer
2,TR_0002,forest,fog,3,2,1.722279,3065.318986,80.265380,12,259.159918,2,2,0.427527,0.636984,490.354140,10.000000,3.762810,18.799661,0.986697,-5.754699,0.000000,-8.351089,14.195603,13.897243,sprinter
3,TR_0003,forest,fog,4,4,1.603709,3237.298212,76.131047,11,247.686196,3,1,0.487792,0.730638,556.248792,10.000000,2.974647,21.146474,1.197847,-8.658303,4.695742,-0.860512,13.064054,9.994055,sprinter
4,TR_0004,urban,clear,2,4,1.806128,3730.035162,72.061673,4,279.860091,7,1,0.687699,0.892439,376.424639,33.829585,6.206864,32.408799,4.544147,-7.942540,11.964464,4.821596,9.304336,1.649628,explorer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,TR_1195,urban,clear,2,4,0.994848,2099.864394,68.468074,3,271.281405,3,3,0.655545,0.852638,698.526981,69.515027,7.801644,28.340666,3.028549,-7.886403,9.686859,9.110390,6.620824,4.064095,explorer
1196,TR_1196,forest,windy,1,2,1.588490,2939.356823,61.791930,5,215.264027,1,3,0.626333,0.760945,426.341872,10.000000,5.506907,33.239349,0.402507,-6.266925,8.410944,1.195784,11.032053,2.512807,explorer
1197,TR_1197,ice,clear,2,3,1.428079,2815.837932,75.243296,6,206.347709,4,2,0.710807,0.816753,461.612654,19.277933,4.858373,27.878398,0.000000,-5.006888,6.423357,2.910617,9.605044,6.687415,explorer
1198,TR_1198,desert,windy,5,3,1.349809,2628.698092,87.725680,9,233.071637,8,4,0.622378,0.819627,607.951880,10.000000,3.442591,28.484507,7.303588,-5.461622,7.214696,-0.469628,10.685672,10.028512,collector


In [ ]:
df_train.dtypes

,0
robot_id,object
arena_type,object
weather,object
difficulty,int64
laps,int64
avg_speed_mps,float64
distance_m,float64
battery_used_pct,float64
collisions,int64
unique_cells,float64


In [ ]:
print('Missing values per column in df_train:')
display(df_train.isnull().sum())

Missing values per column in df_train:


,0
robot_id,0
arena_type,0
weather,0
difficulty,0
laps,0
avg_speed_mps,0
distance_m,0
battery_used_pct,0
collisions,0
unique_cells,0


In [ ]:
print('\nTotal missing values in df_train:')
display(df_train.isnull().sum().sum())


Total missing values in df_train:


np.int64(0)

In [ ]:
df_train['avg_speed_mps'].unique().max()

np.float64(2.641200394995685)

In [ ]:
df_train['arena_type'].nunique()

4

In [ ]:
df_train['arena_type'].value_counts().idxmax()

'forest'

In [ ]:
df_train['items_collected'].unique().max()

np.int64(22)

In [ ]:
def encode_columns(columns):
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=int)
    encoded_train_list = ohe.fit_transform(df_train[columns])
    encoded_test_list = ohe.transform(df_test[columns])

    encoded_df_train = pd.DataFrame(encoded_train_list, columns=ohe.get_feature_names_out())
    encoded_df_test = pd.DataFrame(encoded_test_list, columns=ohe.get_feature_names_out())

    df_train[encoded_df_train.columns] = encoded_df_train
    df_test[encoded_df_test.columns] = encoded_df_test

    df_train.drop(columns, axis=1, inplace=True)
    df_test.drop(columns, axis=1, inplace=True)

def process(df_train, df_test):
    encode_list = ['arena_type', 'weather']
    encode_columns(encode_list)

In [ ]:
process(df_train, df_test)

In [ ]:
output_1 = pd.DataFrame({"robot_id": ["GLOBAL"], "subtaskID": [1], "answer": [4]})
output_2 = pd.DataFrame({"robot_id": ["GLOBAL"], "subtaskID": [2], "answer": [2.641200394995685]})
output_3 = pd.DataFrame({"robot_id": ["GLOBAL"], "subtaskID": [3], "answer": ['forest']})
output_4 = pd.DataFrame({"robot_id": ["GLOBAL"], "subtaskID": [4], "answer": [22]})

In [ ]:

X = df_train.drop(['robot_id', 'strategy_label'], axis=1)
y = df_train['strategy_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
def evaluate_model(model):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    print(f"Train: {f1_score(y_train, y_train_pred, average='macro')}, Test: {f1_score(y_test, y_test_pred, average='macro')}")

In [ ]:
ctb = CatBoostClassifier(verbose=False)
ctb.fit(X_train, y_train)
evaluate_model(ctb)

Train: 1.0, Test: 0.9137098021026593


In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
evaluate_model(lr)

Train: 0.9708273733730328, Test: 0.9274098190932868


In [ ]:
def solve(model):
    X_pred = df_test.drop(['robot_id'], axis=1)
    y_pred = model.predict(X_pred)

    output_5 = pd.DataFrame({"robot_id": df_test['robot_id'], "subtaskID": 5, 'answer': y_pred})
    output = pd.concat([output_1, output_2, output_3, output_4, output_5], axis=0)
    output.to_csv('submission.csv', index=False)

In [ ]:
def solve(model):

    X_pred = df_test.drop(['robot_id'], axis=1)
    X_pred = sc.transform(X_pred)
    y_pred = model.predict(X_pred)

    y_pred = y_pred.flatten() # Ensures y_pred is a 1-dimensional array

    output_5 = pd.DataFrame({"robot_id": df_test['robot_id'], "subtaskID": 5, 'answer': y_pred})
    output = pd.concat([output_1, output_2, output_3, output_4, output_5], axis=0)
    output.to_csv('submission.csv', index=False)

solve(lr)